# Lab 2 – Student Task: Gradient Descent Variants
## Batch GD | Mini-Batch GD | Stochastic GD

**Objective:**  
Implement and compare the three main variants of Gradient Descent for single-variable Linear Regression:
1. **Batch Gradient Descent** – uses the entire dataset to compute each gradient update.
2. **Stochastic Gradient Descent (SGD)** – updates parameters after every single sample.
3. **Mini-Batch Gradient Descent** – balances Batch and SGD by processing fixed-size mini-batches.

For each variant we will:
- Implement the algorithm
- Visualise convergence curves and regression fits
- Evaluate with the R² metric
- Run an **ablation study** to understand the effect of key hyper-parameters

---
## 1. Imports

We need:
- `numpy` for numerical computations (vectorised GD math)
- `matplotlib.pyplot` for visualisations
- `sklearn.metrics.r2_score` to evaluate the fit quality

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.metrics import r2_score

> **Why these libraries?**  
> `numpy` provides fast vectorised operations so we can express the GD update rule
> `θ ← θ − α · (1/m) Xᵀ(Xθ − y)` in a single line instead of nested Python loops.
> `matplotlib` lets us inspect the *loss landscape* and regression line visually – crucial
> to diagnose whether the learning rate is too large (oscillation) or too small (slow convergence).

---
## 2. Dataset Generation

We generate a *noise-free* synthetic dataset where the true relationship is:

$$y = -2x + 1$$

Using a noise-free dataset lets us verify that every algorithm recovers **exactly** θ₀ = 1 and θ₁ = −2.

In [ ]:
# 50 evenly-spaced points between 0 and 20
X = np.linspace(0, 20, 50)  # 50 evenly-spaced points

# True parameters
a, b = -2, 1          # slope, intercept
y = a * X + b

print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"X range: [{X.min():.1f}, {X.max():.1f}]")
print(f"y range: [{y.min():.1f}, {y.max():.1f}]")

> **Data note:** `np.linspace(0, 20)` defaults to 50 evenly-spaced points.
> A small, noise-free dataset is ideal for pedagogical purposes – we can perfectly verify
> the recovered parameters. In practice you would add Gaussian noise
> (e.g. `y += np.random.normal(0, σ, m)`) to simulate real-world measurements.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(X, y, color='steelblue', linewidth=2)
plt.xlabel('X')
plt.ylabel('y')
plt.title('Ground-Truth: y = -2X + 1')
plt.grid(True)
plt.show()

> The perfect linear relationship confirms our data is noise-free.
> GD should converge to θ₀ ≈ 1, θ₁ ≈ −2 with R² ≈ 1.0.

---
## 3. Batch Gradient Descent

### Algorithm
In **Batch GD** every update uses the *full training set*:

$$\theta \leftarrow \theta - \alpha \cdot \frac{1}{m} X^T (X\theta - y)$$

| Pros | Cons |
|---|---|
| Stable, smooth convergence | Slow for large datasets (full pass each step) |
| Exact gradient direction | Cannot be used for online learning |

In [ ]:
def batch_gd(X, y, alpha, max_iter, tol=1e-3):
    """
    Batch Gradient Descent for single-variable Linear Regression.

    Parameters
    ----------
    X        : 1-D array of input features
    y        : 1-D array of target values
    alpha    : learning rate
    max_iter : maximum number of iterations (epochs)
    tol      : gradient norm threshold for early stopping

    Returns
    -------
    theta         : final parameter vector [θ₀, θ₁]
    y_pred        : predictions at convergence
    total_cost    : list of MSE loss at every iteration
    theta_history : list of [θ₀, θ₁] at every iteration
    """
    m = len(y)
    # Add bias column: X_b = [1, x]
    X_b = np.c_[np.ones(m), X]
    theta = np.zeros((2, 1))
    y = y.reshape(-1, 1)

    total_cost = []
    theta_history = []

    for i in range(max_iter):
        y_pred   = X_b @ theta
        error    = y_pred - y
        cost     = (1 / (2 * m)) * np.sum(error ** 2)   # MSE
        gradients = (1 / m) * X_b.T @ error

        # Early stopping: stop when gradient is negligible
        if np.linalg.norm(gradients) < tol:
            print(f'  Converged at iteration {i + 1} | loss={cost:.6f}')
            break

        theta -= alpha * gradients
        total_cost.append(cost)
        theta_history.append(theta.flatten().copy())

    return theta, y_pred, total_cost, theta_history

### Implementation Notes

- **Bias augmentation** (`X_b = [1 | X]`): prepending a column of ones allows us to treat
  θ₀ (intercept) and θ₁ (slope) uniformly inside one matrix multiplication.
- **MSE loss**: `(1/2m) Σ(ŷ − y)²` – the ½ simplifies the gradient derivative.
- **Gradient**: `(1/m) Xᵀ(Xθ − y)` – the exact gradient of MSE with respect to θ.
- **Early stopping** on gradient norm: when `‖∇J‖ < tol`, parameters have essentially stopped
  changing, so further iterations waste compute.

### Visualisation Utilities

We define reusable plotting helpers so the same charts can be generated
for every experiment without code duplication.

In [ ]:
def plot_loss(costs, title='Loss vs Iterations'):
    """Plot the loss curve over iterations / epochs."""
    plt.figure(figsize=(9, 4))
    plt.plot(range(len(costs)), costs, '-o', markersize=4)
    plt.xlabel('Iteration'); plt.ylabel('MSE Loss')
    plt.title(title); plt.grid(True); plt.show()


def plot_theta_vs_loss(theta_hist, costs, title='Theta vs Loss'):
    """Plot θ₀ and θ₁ trajectories against the loss."""
    arr = np.array(theta_hist)
    plt.figure(figsize=(9, 4))
    plt.plot(arr[:, 0], costs, label='θ₀')
    plt.plot(arr[:, 1], costs, label='θ₁')
    plt.xlabel('Theta value'); plt.ylabel('Loss')
    plt.title(title); plt.legend(); plt.grid(True); plt.show()


def plot_regression_history(X, y, theta_hist, title='Regression Lines'):
    """Show how the regression line evolves through all iterations."""
    plt.figure(figsize=(9, 5))
    for t in theta_hist:
        plt.plot(X, t[0] + t[1] * X, color='red', alpha=0.1)
    plt.scatter(X, y, color='steelblue', s=25, zorder=5, label='Data')
    plt.xlabel('X'); plt.ylabel('y')
    plt.title(title); plt.legend(); plt.grid(True); plt.show()


def plot_best_line(X, y, y_pred, title='Best-Fit Regression Line'):
    """Overlay the final best-fit line on the data."""
    plt.figure(figsize=(9, 5))
    plt.scatter(X, y, color='steelblue', label='Actual', s=30)
    plt.plot(X, y_pred, color='crimson', linewidth=2, label='Predicted')
    plt.xlabel('X'); plt.ylabel('y')
    plt.title(title); plt.legend(); plt.grid(True); plt.show()

### 3.1 Run Batch GD with α = 0.0005, 300 iterations

In [ ]:
theta_bgd, y_pred_bgd, cost_bgd, hist_bgd = batch_gd(X, y, alpha=0.0005, max_iter=300)
print(f'Final θ₀={theta_bgd[0][0]:.4f}, θ₁={theta_bgd[1][0]:.4f}')
print(f'R² = {r2_score(y, y_pred_bgd):.6f}')

> **Expected results:** θ₀ ≈ 1, θ₁ ≈ −2, R² ≈ 1.0 (near-perfect fit on noise-free data).

In [ ]:
plot_loss(cost_bgd, 'Batch GD – Loss vs Iterations (α=0.0005)')

> The **smooth, monotonically decreasing** loss curve is the hallmark of Batch GD.
> Every step moves in exactly the direction of steepest descent, making the curve noise-free.

In [ ]:
plot_theta_vs_loss(hist_bgd, cost_bgd, 'Batch GD – Theta vs Loss (α=0.0005)')

> Both θ₀ and θ₁ converge monotonically. The *horizontal* spread shows how far each
> parameter travels before convergence.

In [ ]:
plot_regression_history(X, y, hist_bgd, 'Batch GD – All Regression Lines (α=0.0005)')

> Each semi-transparent red line is one iteration's hypothesis. The lines fan in toward
> the true fit (parallel to the blue data points).

In [ ]:
plot_best_line(X, y, y_pred_bgd, 'Batch GD – Best-Fit Line (α=0.0005)')

---
## 🔬 Ablation Study 1 – Effect of Learning Rate on Batch GD

**Why this ablation?**  
The learning rate α is the most sensitive hyper-parameter in GD.  
Too large → overshooting / divergence. Too small → slow convergence.  
By systematically sweeping α we can identify the *Goldilocks zone* and build intuition
about the loss landscape curvature.

We test: **α ∈ {0.00005, 0.0001, 0.0005, 0.005}** with a fixed budget of 500 iterations.

In [ ]:
alphas      = [0.00005, 0.0001, 0.0005, 0.005]
colors      = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
results_bgd = {}

plt.figure(figsize=(12, 5))
for alpha, color in zip(alphas, colors):
    _, y_pred, costs, _ = batch_gd(X, y, alpha=alpha, max_iter=500)
    results_bgd[alpha] = {'r2': r2_score(y, y_pred), 'iters': len(costs)}
    plt.plot(costs, label=f'α={alpha}', color=color)

plt.xlabel('Iteration'); plt.ylabel('MSE Loss')
plt.title('Ablation 1 – Learning Rate Effect on Batch GD Convergence')
plt.legend(); plt.grid(True); plt.yscale('log'); plt.show()

print('\nSummary:')
print(f'{"Alpha":>10} | {"R²":>10} | {"Iterations":>12}')
print('-' * 38)
for alpha, res in results_bgd.items():
    print(f'{alpha:>10} | {res["r2"]:>10.6f} | {res["iters"]:>12}')

### Ablation 1 – Observations

| α | Behaviour |
|---|---|
| 0.00005 | Very slow – still converging at iteration 500 |
| 0.0001 | Slower but stable |
| **0.0005** | **Sweet spot** – fast convergence, stable |
| 0.005 | Risks overshooting at early iterations |

**Takeaway:** α = 0.0005 achieves the best balance between speed and stability on this dataset.
A practical heuristic is to start with a moderate α and halve it if the loss oscillates.

---
## 🔬 Ablation Study 2 – Convergence Tolerance in Batch GD

**Why this ablation?**  
The early-stopping criterion `‖∇J‖ < tol` determines when we declare convergence.
A tight tolerance (`tol=1e-6`) requires more iterations but gives a more precise solution;
a loose tolerance (`tol=1e-2`) exits early but may leave significant residual error.

We compare: **tol ∈ {1e-2, 1e-3, 1e-4, 1e-6}** at fixed α = 0.0005.

In [ ]:
tolerances = [1e-2, 1e-3, 1e-4, 1e-6]

print(f'{"Tolerance":>12} | {"Iterations":>12} | {"R²":>10} | {"Final Loss":>12}')
print('-' * 54)
for tol in tolerances:
    _, y_pred, costs, _ = batch_gd(X, y, alpha=0.0005, max_iter=5000, tol=tol)
    final_loss = costs[-1] if costs else float('nan')
    r2 = r2_score(y, y_pred)
    print(f'{tol:>12.0e} | {len(costs):>12} | {r2:>10.6f} | {final_loss:>12.6f}')

### Ablation 2 – Observations

- A tolerance of `1e-3` (default) already gives R² ≈ 1.0 on noise-free data,
  but tighter tolerances squeeze out slightly smaller final losses.
- On **noisy** real-world data, a loose tolerance often generalises *better* because it
  prevents over-fitting to noise (acting as a form of implicit regularisation).
- **Recommendation:** use `tol=1e-4` as a good default; tighten only when precision is critical.

---
## 4. Stochastic Gradient Descent (SGD)

### Algorithm
SGD updates θ after **every single sample**, so each epoch contains *m* updates:

$$\theta \leftarrow \theta - \alpha \cdot \nabla J(\theta; x^{(i)}, y^{(i)})$$

| Pros | Cons |
|---|---|
| Very fast iterations (1 sample at a time) | Noisy loss curve – erratic convergence |
| Can escape shallow local minima | Requires more careful LR tuning |
| Suitable for online/stream learning | Never fully 'settles' at the minimum |

In [ ]:
def stochastic_gd(X, y, lr, epochs, tol=1e-3, seed=101):
    """
    Stochastic Gradient Descent for single-variable Linear Regression.

    Parameters
    ----------
    X      : 1-D array of input features
    y      : 1-D array of target values
    lr     : learning rate
    epochs : maximum number of passes over the full dataset
    tol    : epoch-level loss change threshold for early stopping
    seed   : random seed for reproducibility

    Returns
    -------
    theta_hist_0   : list of θ₀ after every sample update
    theta_hist_1   : list of θ₁ after every sample update
    epoch_losses   : list of per-epoch MSE (for convergence check)
    iter_losses    : list of per-sample loss (for noisy loss plot)
    opt_theta_0    : final θ₀
    opt_theta_1    : final θ₁
    """
    m = len(y)
    X_b = np.c_[np.ones(m), X]
    theta = np.zeros((2, 1))
    y = y.reshape(-1, 1)
    np.random.seed(seed)

    theta_hist_0, theta_hist_1 = [], []
    epoch_losses, iter_losses  = [], []

    for epoch in range(epochs):
        # Shuffle data each epoch to avoid cyclic updates
        idx = np.random.permutation(m)
        X_s, y_s = X_b[idx], y[idx]

        for i in range(m):
            xi, yi = X_s[i:i+1], y_s[i:i+1]
            error  = xi @ theta - yi
            grad   = xi.T @ error
            theta -= lr * grad
            theta_hist_0.append(theta[0, 0])
            theta_hist_1.append(theta[1, 0])
            iter_losses.append(0.5 * float(np.sum(error ** 2)))

        # Epoch-level loss for convergence criterion
        epoch_err  = X_b @ theta - y
        epoch_loss = float((1 / (2 * m)) * np.sum(epoch_err ** 2))
        epoch_losses.append(epoch_loss)

        if epoch > 0 and abs(epoch_losses[-2] - epoch_loss) < tol:
            print(f'  Converged at epoch {epoch + 1}')
            break

    return theta_hist_0, theta_hist_1, epoch_losses, iter_losses, theta[0, 0], theta[1, 0]

### Implementation Notes – SGD

- **Per-sample gradient** `xᵢᵀ(xᵢθ − yᵢ)`: no 1/m factor since we have only one sample.
- **Shuffle each epoch**: critical to avoid the model 'memorising' the data order instead
  of learning the underlying distribution.
- **Epoch-level convergence check**: checking convergence per-sample is unreliable because
  the loss is noisy. We check *after* each full epoch when the noise has averaged out.

### 4.1 Run SGD with α = 0.0005, 300 epochs

In [ ]:
th0, th1, ep_loss, it_loss, opt_t0, opt_t1 = stochastic_gd(X, y, lr=0.0005, epochs=300)

y_pred_sgd = opt_t0 + opt_t1 * X
print(f'Final θ₀={opt_t0:.4f}, θ₁={opt_t1:.4f}')
print(f'R² = {r2_score(y, y_pred_sgd):.6f}')

In [ ]:
# Show first 150 iteration-level losses to see the stochastic noise
plt.figure(figsize=(12, 4))
plt.plot(it_loss[:150], alpha=0.7)
plt.xlabel('Sample update (iteration)'); plt.ylabel('Sample MSE')
plt.title('SGD – Iteration-Level Loss (α=0.0005, first 150 updates)')
plt.grid(True); plt.show()

# Epoch-level loss is much smoother
plot_loss(ep_loss, 'SGD – Epoch-Level Loss (α=0.0005)')

> **Notice the two loss views:**
> - *Iteration-level*: noisy because each sample has a different gradient direction.
> - *Epoch-level*: much smoother – the average over all samples converges steadily.
> In practice we monitor the epoch-level loss to decide on early stopping.

In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(X, y, color='steelblue', s=30, label='Data')
plt.plot(X, y_pred_sgd, color='crimson', linewidth=2, label='SGD fit')
plt.xlabel('X'); plt.ylabel('y')
plt.title('SGD – Best-Fit Regression Line (α=0.0005)')
plt.legend(); plt.grid(True); plt.show()

---
## 🔬 Ablation Study 3 – Learning Rate Effect on SGD

**Why this ablation?**  
SGD is considerably more sensitive to α than Batch GD because the *per-sample*
gradient is much noisier. A large α can cause the loss to *diverge* almost
immediately. This ablation shows the typical failure modes.

We test: **α ∈ {0.00007, 0.0001, 0.0005, 0.001}** for 300 epochs.

In [ ]:
sgd_alphas = [0.00007, 0.0001, 0.0005, 0.001]
colors_sgd = ['#9467bd', '#8c564b', '#e377c2', '#17becf']

plt.figure(figsize=(12, 5))
print(f'{"Alpha":>10} | {"R²":>10} | {"Epochs":>8}')
print('-' * 36)
for alpha, color in zip(sgd_alphas, colors_sgd):
    _, _, ep_loss, _, opt_t0, opt_t1 = stochastic_gd(X, y, lr=alpha, epochs=300)
    r2 = r2_score(y, opt_t0 + opt_t1 * X)
    print(f'{alpha:>10} | {r2:>10.6f} | {len(ep_loss):>8}')
    plt.plot(ep_loss, label=f'α={alpha}', color=color)

plt.xlabel('Epoch'); plt.ylabel('Epoch MSE')
plt.title('Ablation 3 – SGD: Learning Rate Effect on Epoch Loss')
plt.legend(); plt.grid(True); plt.yscale('log'); plt.show()

### Ablation 3 – Observations

- Very small α (0.00007) needs more epochs to reach the same loss level.
- α = 0.0005 converges quickly with stable epoch losses.
- Larger α (0.001) may show oscillation in the epoch loss, a sign of overshooting.
- **Key insight:** SGD needs a *smaller* optimal α than Batch GD for the same dataset
  because the effective step is noisier and can overshoot more easily.

---
## 5. Mini-Batch Gradient Descent

### Algorithm
Mini-Batch GD divides the training set into mini-batches of size *B* and performs
one gradient update per batch:

$$\theta \leftarrow \theta - \alpha \cdot \frac{1}{B} X_{\text{batch}}^T (X_{\text{batch}}\theta - y_{\text{batch}})$$

| Pros | Cons |
|---|---|
| Balances stability (Batch) and speed (SGD) | Extra hyper-parameter: batch size |
| Vectorised computation → GPU-friendly | Tuning batch size adds complexity |
| Moderately smooth loss curve | |

In [ ]:
def mini_batch_gd(X, y, lr, epochs, batch_size, tol=1e-3, seed=101):
    """
    Mini-Batch Gradient Descent for single-variable Linear Regression.

    Parameters
    ----------
    X          : 1-D array of input features
    y          : 1-D array of target values
    lr         : learning rate
    epochs     : maximum number of passes over the full dataset
    batch_size : number of samples per mini-batch
    tol        : epoch-level loss change threshold for early stopping
    seed       : random seed for reproducibility

    Returns
    -------
    theta_hist_0    : list of θ₀ after every mini-batch update
    theta_hist_1    : list of θ₁ after every mini-batch update
    batch_losses    : list of per-batch MSE
    epoch_losses    : list of per-epoch MSE
    theta_0         : final θ₀
    theta_1         : final θ₁
    """
    m = len(y)
    X_b = np.c_[np.ones(m), X]
    theta = np.zeros((2, 1))
    y = y.reshape(-1, 1)
    np.random.seed(seed)

    theta_hist_0, theta_hist_1 = [], []
    batch_losses, epoch_losses  = [], []

    for epoch in range(epochs):
        idx = np.random.permutation(m)
        X_s, y_s = X_b[idx], y[idx]

        for start in range(0, m, batch_size):
            xi = X_s[start:start + batch_size]
            yi = y_s[start:start + batch_size]
            b  = len(xi)

            error = xi @ theta - yi
            grad  = (1 / b) * xi.T @ error
            theta -= lr * grad

            theta_hist_0.append(theta[0, 0])
            theta_hist_1.append(theta[1, 0])
            batch_losses.append(float((1 / (2 * b)) * np.sum(error ** 2)))

        # Epoch-level check
        epoch_err  = X_b @ theta - y
        epoch_loss = float((1 / (2 * m)) * np.sum(epoch_err ** 2))
        epoch_losses.append(epoch_loss)

        if epoch > 0 and abs(epoch_losses[-2] - epoch_loss) < tol:
            print(f'  Converged at epoch {epoch + 1}')
            break

    return theta_hist_0, theta_hist_1, batch_losses, epoch_losses, theta[0, 0], theta[1, 0]

### Implementation Notes – Mini-Batch GD

- **Batch size B** controls the variance/bias trade-off of the gradient estimate:
  - B = 1 → SGD (maximum noise, fastest iteration)
  - B = m → Batch GD (zero noise, slowest iteration)
  - 16 ≤ B ≤ 512 → typical deep-learning range
- **Epoch-level convergence check**: same rationale as SGD — per-batch losses are too
  noisy for a reliable stopping criterion.

### 5.1 Run Mini-Batch GD – batch_size=5, α=0.0005, 300 epochs

In [ ]:
th0_mb, th1_mb, bl_mb, el_mb, t0_mb, t1_mb = mini_batch_gd(
    X, y, lr=0.0005, epochs=300, batch_size=5
)
y_pred_mb = t0_mb + t1_mb * X
print(f'Final θ₀={t0_mb:.4f}, θ₁={t1_mb:.4f}')
print(f'R² = {r2_score(y, y_pred_mb):.6f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].plot(bl_mb[:100], '-o', markersize=3)
axes[0,0].set_title('Batch-level Loss (first 100 batches)')
axes[0,0].set_xlabel('Batch update'); axes[0,0].set_ylabel('MSE')
axes[0,0].grid(True)

axes[0,1].plot(el_mb, '-o', markersize=4)
axes[0,1].set_title('Epoch-level Loss')
axes[0,1].set_xlabel('Epoch'); axes[0,1].set_ylabel('MSE')
axes[0,1].grid(True)

axes[1,0].plot(th0_mb, bl_mb, alpha=0.5); axes[1,0].plot(th1_mb, bl_mb, alpha=0.5)
axes[1,0].set_title('Theta vs Batch Loss')
axes[1,0].set_xlabel('Theta value'); axes[1,0].set_ylabel('Loss')
axes[1,0].grid(True)

axes[1,1].scatter(X, y, color='steelblue', s=30, label='Data')
axes[1,1].plot(X, y_pred_mb, color='crimson', linewidth=2, label='Mini-Batch fit')
axes[1,1].set_title('Best-Fit Line (batch_size=5)')
axes[1,1].set_xlabel('X'); axes[1,1].set_ylabel('y')
axes[1,1].legend(); axes[1,1].grid(True)

plt.tight_layout(); plt.show()

> The batch-level loss is noticeably *smoother* than SGD's per-sample loss
> but still shows some variance, unlike the perfectly smooth Batch GD loss.

---
## 🔬 Ablation Study 4 – Batch Size Effect on Mini-Batch GD

**Why this ablation?**  
Batch size is *unique* to Mini-Batch GD (it doesn't apply to Batch or SGD).
Smaller batches give noisier gradient estimates (closer to SGD) and can escape
local minima, while larger batches give smoother convergence (closer to Batch GD).

We test: **B ∈ {1, 5, 10, 25, 50}** at fixed α = 0.0005 and 300 epochs.
Note: B=1 reproduces SGD; B=50 (= dataset size) reproduces Batch GD.

In [ ]:
batch_sizes = [1, 5, 10, 25, 50]
colors_bs   = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

print(f'{"Batch Size":>12} | {"R²":>10} | {"Epochs":>8} | {"Batch Updates":>14}')
print('-' * 54)
for bs, color in zip(batch_sizes, colors_bs):
    _, _, bl, el, t0, t1 = mini_batch_gd(X, y, lr=0.0005, epochs=300, batch_size=bs)
    r2 = r2_score(y, t0 + t1 * X)
    print(f'{bs:>12} | {r2:>10.6f} | {len(el):>8} | {len(bl):>14}')
    ax1.plot(el, label=f'B={bs}', color=color)          # epoch-level
    ax2.plot(bl[:200], alpha=0.6, label=f'B={bs}', color=color)  # batch-level (first 200)

ax1.set_title('Epoch-Level Loss vs Batch Size'); ax1.set_xlabel('Epoch')
ax1.set_ylabel('MSE'); ax1.legend(); ax1.grid(True)

ax2.set_title('Batch-Level Loss (first 200 updates)'); ax2.set_xlabel('Batch update')
ax2.set_ylabel('MSE'); ax2.legend(); ax2.grid(True)

plt.tight_layout(); plt.show()

### Ablation 4 – Observations

| Batch Size | Noise level | Convergence speed | Memory |
|---|---|---|---|
| 1 (SGD) | Highest | Potentially fast, erratic | Lowest |
| 5–10 | Moderate | Fast | Low |
| 25–50 | Low | Stable but more steps needed | Higher |
| 50 (≈ Batch) | Lowest | Smooth but slow per-update | Highest |

**Takeaway:** A batch size in the range [5, 25] typically offers the best
convergence–stability trade-off on this 50-sample dataset.  
In deep learning, powers of 2 (32, 64, 128) are preferred for GPU efficiency.

---
## 6. Algorithm Comparison: Batch GD vs SGD vs Mini-Batch GD

We now run all three algorithms with the same hyper-parameters (α=0.0005, 300 epochs)
and compare their convergence on the **epoch-level loss**.

In [ ]:
# Run Batch GD
_, y_pred_b, cost_b, _ = batch_gd(X, y, alpha=0.0005, max_iter=300)
print(f'Batch GD   R^2={r2_score(y, y_pred_b):.6f}')


In [ ]:
# Run SGD
_, _, ep_sgd, _, t0_s, t1_s = stochastic_gd(X, y, lr=0.0005, epochs=300)
y_pred_s = t0_s + t1_s * X
print(f'SGD        R^2={r2_score(y, y_pred_s):.6f}')


In [ ]:
# Run Mini-Batch GD (batch_size=10)
_, _, _, ep_mb, t0_m, t1_m = mini_batch_gd(X, y, lr=0.0005, epochs=300, batch_size=10)
y_pred_m = t0_m + t1_m * X
print(f'Mini-Batch R^2={r2_score(y, y_pred_m):.6f}')


In [ ]:
# Plot epoch-level loss comparison (log scale for clarity)
plt.figure(figsize=(12, 5))
plt.plot(cost_b,  label='Batch GD', linewidth=2)
plt.plot(ep_sgd,  label='SGD', linewidth=2, linestyle='--')
plt.plot(ep_mb,   label='Mini-Batch B=10', linewidth=2, linestyle=':')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('Algorithm Comparison - Epoch-Level Loss (alpha=0.0005)')
plt.legend(); plt.grid(True); plt.yscale('log'); plt.show()
print('R^2 Summary:')
print(f'  Batch GD:   {r2_score(y, y_pred_b):.6f}')
print(f'  SGD:        {r2_score(y, y_pred_s):.6f}')
print(f'  Mini-Batch: {r2_score(y, y_pred_m):.6f}')


### Best-Fit Lines – All Three Methods

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(X, y, color='steelblue', s=30, zorder=5, label='Data')
plt.plot(X, y_pred_b, color='green',  linewidth=2, label='Batch GD')
plt.plot(X, y_pred_s, color='orange', linewidth=2, linestyle='--', label='SGD')
plt.plot(X, y_pred_m, color='red',    linewidth=2, linestyle=':',  label='Mini-Batch')
plt.xlabel('X'); plt.ylabel('y')
plt.title('Best-Fit Lines – Algorithm Comparison')
plt.legend(); plt.grid(True); plt.show()

### Summary Table

| Algorithm | Gradient Estimate | Loss Curve | Best For |
|---|---|---|---|
| **Batch GD** | Exact (full dataset) | Smooth, monotone | Small datasets, convex problems |
| **SGD** | Noisy (single sample) | Erratic (iteration-level) | Online / streaming data, large datasets |
| **Mini-Batch GD** | Approximate (batch) | Semi-smooth | Deep learning, GPU-accelerated training |

All three converge to R² ≈ 1.0 on this noise-free dataset,
confirming that each algorithm correctly recovers the true parameters θ₀=1, θ₁=−2.

---
## 🔬 Ablation Study 5 – Noisy Data: Robustness of Each Algorithm

**Why this ablation?**  
All previous experiments used *noise-free* data. Real-world datasets always contain noise.
We now add Gaussian noise (σ = 3) to y and compare how each algorithm's R² and final
loss are affected. This reveals which algorithm generalises most robustly.

**Hypothesis:** Because the true signal is linear, all algorithms should still recover
approximately the same θ, but SGD's noisy updates may help avoid over-fitting to noise.

In [ ]:
np.random.seed(42)
y_noisy = y + np.random.normal(0, 3, size=y.shape)   # add Gaussian noise σ=3

# ── Run all algorithms on noisy data ──────────────────────────────────────
theta_b_n, y_pred_b_n, cost_b_n, _ = batch_gd(X, y_noisy, alpha=0.0005, max_iter=300)
t0_b_n, t1_b_n = float(theta_b_n.flatten()[0]), float(theta_b_n.flatten()[1])

_, _, ep_s_n, it_s_n, t0_s_n, t1_s_n = stochastic_gd(X, y_noisy, lr=0.0005, epochs=300)
y_pred_s_n = t0_s_n + t1_s_n * X

_, _, _, ep_m_n, t0_m_n, t1_m_n = mini_batch_gd(
    X, y_noisy, lr=0.0005, epochs=300, batch_size=10)
y_pred_m_n = t0_m_n + t1_m_n * X

# ── Compare ───────────────────────────────────────────────────────────────
print('Algorithm Performance on Noisy Data (σ=3):')
print(f'{"Algorithm":>20} | {"R² (noisy)":>12} | {"θ₀":>8} | {"θ₁":>8}')
print('-' * 58)
print(f'{"Batch GD":>20} | {r2_score(y_noisy, y_pred_b_n):>12.4f} | {t0_b_n:>8.3f} | {t1_b_n:>8.3f}')
print(f'{"SGD":>20} | {r2_score(y_noisy, y_pred_s_n):>12.4f} | {t0_s_n:>8.3f} | {t1_s_n:>8.3f}')
print(f'{"Mini-Batch (B=10)":>20} | {r2_score(y_noisy, y_pred_m_n):>12.4f} | {t0_m_n:>8.3f} | {t1_m_n:>8.3f}')

# ── Visual comparison ─────────────────────────────────────────────────────
plt.figure(figsize=(10, 5))
plt.scatter(X, y_noisy, color='steelblue', s=30, alpha=0.7, zorder=5, label='Noisy data')
plt.plot(X, y, color='black',  linewidth=1.5, linestyle='--', label='True line')
plt.plot(X, y_pred_b_n.flatten(), color='green',  linewidth=2, label='Batch GD')
plt.plot(X, y_pred_s_n, color='orange', linewidth=2, linestyle='--', label='SGD')
plt.plot(X, y_pred_m_n, color='red',    linewidth=2, linestyle=':',  label='Mini-Batch')
plt.xlabel('X'); plt.ylabel('y')
plt.title('Noisy Data Ablation – Recovered Regression Lines')
plt.legend(); plt.grid(True); plt.show()


### Ablation 5 – Observations

- All three algorithms fit the noisy data to approximately the same quality,
  which is expected on a small, purely linear problem.
- With stronger noise or more complex models, the regularisation implicit in
  early-stopping SGD (or momentum) can yield better generalisation than exact
  gradient updates.
- **Key takeaway:** the choice of GD variant matters more for *speed* and *scalability*
  than for final accuracy on simple linear regression.

---
## 7. Conclusions

| Question | Answer |
|---|---|
| Which algorithm is most stable? | **Batch GD** – smooth, monotone loss |
| Which is fastest per-epoch? | **SGD** – single-sample update |
| Which is the practical default? | **Mini-Batch GD** – used by all major DL frameworks |
| Most sensitive hyper-param? | **Learning rate α** – most important for all variants |
| Second most important? | **Batch size B** – specific to Mini-Batch GD |

### What we learned from the ablation studies

1. **Ablation 1 (LR for Batch GD):** α = 0.0005 is in the sweet spot for this 50-sample
   dataset. Smaller α requires significantly more iterations; larger α risks instability.

2. **Ablation 2 (Tolerance):** `tol=1e-3` already achieves near-perfect R² on noise-free data.
   On real noisy data a looser tolerance can improve generalisation.

3. **Ablation 3 (LR for SGD):** SGD needs a carefully tuned, typically smaller α than Batch GD
   because single-sample gradients are noisy and can cause overshooting.

4. **Ablation 4 (Batch Size):** Batch sizes B ∈ [5, 25] balance noise and speed best on this
   dataset. Powers of 2 (16, 32) are preferred in deep learning for hardware alignment.

5. **Ablation 5 (Noisy Data):** All variants recover approximately the true θ even under
   Gaussian noise, confirming the robustness of GD-based linear regression.